In [2]:
#验证noise_utils.py核心函数
import sys
from pathlib import Path
import numpy as np

root = Path.cwd().resolve()

if root.name.lower() == "notebooks":
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.noise_utils import (
    measure_power,
    measure_snr_db,
    add_gaussian_noise_at_snr,
    protect_pair_peak
)

#生成1s、16kHz的单音信号
sr = 16000
t = np.arange(sr) / sr
clean = 0.5 * np.sin(2 * np.pi * 440 * t)

#目标10dB加噪
noisy, noise = add_gaussian_noise_at_snr(
    clean,
    target_snr_db=10,
    seed=42
)

print("clean功率:",measure_power(clean))
print("noise功率:",measure_power(noise))
print("实测SNR:", measure_snr_db(clean, noisy))

#相同seed再次生成
noisy_again, _ = add_gaussian_noise_at_snr(
    clean,
    target_snr_db=10,
    seed=42
)

print(
    "相同seed结果一致:",
    np.array_equal(noisy, noisy_again)
)

clean功率: 0.12499999999999983
noise功率: 0.012499999999999982
实测SNR: 10.0
相同seed结果一致: True


In [4]:
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display

manifest_path = root / "metadata" / "manifest.csv"

audio_root = (
    root / "data" / "raw"
    / "common_voice_beijing26"
    / "zh-CN-beijing"
)

manifest = pd.read_csv(manifest_path)

train = manifest.loc[
    manifest["split"] == "train"
].copy()

# 本次先使用固定的一条train音频
row = train.iloc[0]

p = Path(str(row["audio_path"]))

if p.is_absolute():
    candidates = [p]
else:
    candidates = [
        root / p,
        audio_root / p,
        audio_root / "clips" / p
    ]

audio_path = next(
    (path for path in candidates if path.is_file()),
    None
)

if audio_path is None:
    raise FileNotFoundError(
        f"找不到音频文件：{row['audio_path']}"
    )

audio, sr = sf.read(
    audio_path,
    dtype="float32",
    always_2d=True
)

#soundfile返回(T,C)，沿声道维取均值
clean = audio.mean(axis=1, dtype=np.float64)
parent_uid = row["parent_uid"]

#生成10dB合成含噪语音
noisy, noise = add_gaussian_noise_at_snr(
    clean, 
    target_snr_db=10,
    seed=2026
)

print("parent_uid:",parent_uid)
print("原始采样率:",sr)
print("波形形状:",clean.shape)
print("加噪前实测SNR:",measure_snr_db(clean,noisy))

#峰值保护
clean_safe, noisy_safe, scale = protect_pair_peak(
    clean, noisy, ceiling=0.99
)

print("共同缩放系数：",scale)
print(
    "共同缩放后实测SNR:",
    measure_snr_db(clean_safe,noisy_safe)
)

print("clean峰值:", np.max(np.abs(clean_safe)))
print("noisy峰值:", np.max(np.abs(noisy_safe)))

print("\nClean:")
display(Audio(clean_safe, rate=sr, normalize=False))

print("\nSynthetic noisy 10dB:")
display(Audio(noisy_safe, rate=sr, normalize=False))

parent_uid: common_voice_zh-CN_27897382
原始采样率: 32000
波形形状: (202752,)
加噪前实测SNR: 10.0
共同缩放系数： 1.0
共同缩放后实测SNR: 10.0
clean峰值: 0.40968191623687744
noisy峰值: 0.41668402868865667

Clean:



Synthetic noisy 10dB:


In [6]:
output_dir = root / "data" / "derived" / "w05_snr"
output_dir.mkdir(parents=True, exist_ok=True)

clean_path = output_dir / "demo_clean.wav"
noisy_path = output_dir / "demo_synthetic_10db.wav"

#避免覆盖之前的实验结果
if clean_path.exists() or noisy_path.exists():
    raise FileExistsError(
        "实验输出文件已存在，请先检查，不要直接覆盖"
    )

#WAV FLOAT格式，避免整数PCM的额外量化影响
sf.write(
    clean_path,
    clean_safe.astype(np.float32),
    sr,
    format="WAV",
    subtype="FLOAT"
)

sf.write(
    noisy_path,
    noisy_safe.astype(np.float32),
    sr,
    format="WAV",
    subtype="FLOAT"
)

#回读音频
clean_read, sr1 = sf.read(clean_path)
noisy_read, sr2 = sf.read(noisy_path)

assert sr1 == sr2 == sr
assert clean_read.shape == noisy_read.shape

measured_snr = measure_snr_db(
    clean_read,
    noisy_read
)

peak = max(
    np.max(np.abs(clean_read)),
    np.max(np.abs(noisy_read))
)

print("写盘后实测SNR:", measured_snr)
print("写盘后最大绝对幅值:", peak)

#本次实验的工程验收阈值
assert abs(measured_snr - 10) <= 0.5
assert peak <= 0.99 + 1e-6

print("写盘后SNR与峰值检查通过")

写盘后实测SNR: 10.000000002132317
写盘后最大绝对幅值: 0.41668403148651123
写盘后SNR与峰值检查通过


In [8]:
#生成 results/w05_snr.csv
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import soundfile as sf

from src.noise_utils import measure_snr_db

#1.确定项目路径
root = Path.cwd().resolve()

if root.name.lower() == "notebooks":
    root = root.parent

manifest_path = root / "metadata" / "manifest.csv"
candidate_path = root / "metadata" / "test_candidates.csv"

clean_path = (
    root / "data" / "derived" / "w05_snr"
    / "demo_clean.wav"
)

noisy_path = (
    root / "data" / "derived" / "w05_snr"
    / "demo_synthetic_10db.wav"
)

results_path = root / "results" / "w05_snr.csv"

#2.检查输入文件
for path in [
    manifest_path,
    candidate_path,
    clean_path,
    noisy_path,
]:
    if not path.is_file():
        raise FileNotFoundError(f"找不到文件：{path}")

#3.从manifest读取原始来源信息
manifest = pd.read_csv(
    manifest_path,
    dtype={"parent_uid":str, "split": str}
)

parent_uid = "common_voice_zh-CN_27897382"

source_rows = manifest.loc[
    manifest["parent_uid"].eq(parent_uid)
]

if len(source_rows) != 1:
    raise ValueError(
        f"预期找到1条来源记录，实际找到{len(source_rows)}条"
    )

source_rows = source_rows.iloc[0]
source_split = str(source_rows["split"])

if source_split != "train":
    raise ValueError(
        f"本次实验要求使用train音频，实际为{source_split}"
    )

#4.重新读取保存的音频
clean_read, sr_clean = sf.read(
    clean_path, dtype="float64"
)

noisy_read, sr_noisy = sf.read(
    noisy_path, dtype="float64"
)

if sr_clean != sr_noisy:
    raise ValueError("clean与noisy的采样率不一致")

if clean_read.shape != noisy_read.shape:
    raise ValueError("clean与noisy的波形形状不一致")

#5.测量写盘后的真实SNR与峰值
measured_snr = measure_snr_db(
    clean_read, noisy_read
)

peak = max(
    np.max(np.abs(clean_read)),
    np.max(np.abs(noisy_read)),
)

if abs(measured_snr - 10.0) > 0.5:
    raise ValueError(
        f"实测SNR不符合要求：{measured_snr:.4f} dB"
    )

if peak > 0.99 + 1e-6:
    raise ValueError(f"峰值超过上限：{peak}")

#6.准备两条实验记录
rows = pd.DataFrame([
    {
        "parent_uid": parent_uid,
        "source_split": source_split,
        "source_audio_path": str(source_rows["audio_path"]),
        "condition": "clean",
        "target_snr_db": np.nan,
        "measured_snr_db": np.nan,
        "seed": np.nan,
        "sample_rate": sr_clean,
        "peak_scale": 1.0,
        "output_path": str(clean_path.relative_to(root)),
        "noise_type": "none",
    },
    {
        "parent_uid": parent_uid,
        "source_split": source_split,
        "source_audio_path": str(source_rows["audio_path"]),
        "condition": "synthetic",
        "target_snr_db": 10.0,
        "measured_snr_db": measured_snr,
        "seed": 2026,
        "sample_rate": sr_noisy,
        "peak_scale": 1.0,
        "output_path": str(noisy_path.relative_to(root)),
        "noise_type": "gaussian_white",
    }
])

#7.不覆盖已有试验记录
if results_path.exists():
    raise FileExistsError(
        f"{results_path} 已存在，请先检查现有记录，勿直接覆盖。"
    )

results_path.parent.mkdir(
    parents=True,exist_ok=True
)

#8.写入CSV
rows.to_csv(results_path, index=False)

print("已创建:", results_path)
print("\n实验记录:")
print(rows.to_string(index=False))

已创建: D:\Projects\speech-ai-learning\results\w05_snr.csv

实验记录:
                 parent_uid source_split               source_audio_path condition  target_snr_db  measured_snr_db   seed  sample_rate  peak_scale                                  output_path     noise_type
common_voice_zh-CN_27897382        train common_voice_zh-CN_27897382.mp3     clean            NaN              NaN    NaN        32000         1.0          data\derived\w05_snr\demo_clean.wav           none
common_voice_zh-CN_27897382        train common_voice_zh-CN_27897382.mp3 synthetic           10.0             10.0 2026.0        32000         1.0 data\derived\w05_snr\demo_synthetic_10db.wav gaussian_white


In [9]:
#检查实验表是否符合要求
import pandas as pd
import numpy as np

results = pd.read_csv(results_path)

# 必须恰好有两个条件
assert len(results) == 2
assert set(results["condition"]) == {
    "clean", "synthetic"
}

# 两条记录必须来自同一条train音频
assert results["parent_uid"].nunique() == 1
assert results["source_split"].eq("train").all()

clean_row = results.loc[
    results["condition"] == "clean"
].iloc[0]

synthetic_row = results.loc[
    results["condition"] == "synthetic"
].iloc[0]


# clean的SNR和seed不应伪造
assert pd.isna(clean_row["target_snr_db"])
assert pd.isna(clean_row["measured_snr_db"])
assert pd.isna(clean_row["seed"])

# synthetic的实验参数必须正确
assert synthetic_row["target_snr_db"] == 10.0
assert abs(
    synthetic_row["measured_snr_db"] - 10.0
) <= 0.5

assert synthetic_row["seed"] == 2026
assert synthetic_row["noise_type"] == "gaussian_white"


# 两个输出文件必须真实存在
for path in results["output_path"]:
    assert (root / path).is_file(), f"文件不存在：{path}"

# 检查采样率与峰值缩放系数
assert results["sample_rate"].eq(32000).all()
assert np.allclose(results["peak_scale"], 1.0)

print("模块3实验表检查全部通过")
print("记录数量:", len(results))
print("来源:", results["parent_uid"].iloc[0])
print("来源划分:", results["source_split"].iloc[0])
print(
    "实际SNR:",
    round(float(synthetic_row["measured_snr_db"]), 6),
    "dB",
)
print("随机seed:", int(synthetic_row["seed"]))
print("输出文件均存在:", True)

模块3实验表检查全部通过
记录数量: 2
来源: common_voice_zh-CN_27897382
来源划分: train
实际SNR: 10.0 dB
随机seed: 2026
输出文件均存在: True
